# Ropedia Academy — D9 · Capstone: pixels to world memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/D9.ipynb)

> **Runs the whole curriculum as one pipeline and visualizes it end to end: TSDF geometry fusion converging beside the resulting queryable scene graph.**
>
> 把整门课程作为一条流水线运行，并端到端可视化：TSDF 几何融合收敛，旁边是由此得到的可查询场景图。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/D9

In [ ]:
import numpy as np, networkx as nx, matplotlib.pyplot as plt
np.random.seed(0)

# CAPSTONE: pixels -> world memory. Wire every track into one runnable pipeline.
frames = [{"depth": 1.0 + np.random.randn()*0.05, "label": 2} for _ in range(20)]
poses = [np.eye(4) for _ in frames]                    # 1) SLAM/SfM poses (B3/D2)
tsdf, w, hist = 0.0, 0, []                              # 2) TSDF fusion (D3)
for f in frames:
    tsdf = (w*tsdf + np.clip(f["depth"]-1.0, -1, 1))/(w+1); w += 1; hist.append(tsdf)
obj = int(np.bincount([f["label"] for f in frames]).argmax())   # 3) semantics (D4)
g = nx.DiGraph(); g.add_edge(f"obj{obj}", "table", rel="on"); g.add_edge("table","kitchen", rel="in")
print("object in kitchen?:", nx.has_path(g, f"obj{obj}", "kitchen"))   # 4) scene graph (D5)

# Visualize the pipeline: geometry fusion converging + the resulting scene graph.
fig, ax = plt.subplots(1, 2, figsize=(8.5, 3.5))
ax[0].plot(hist); ax[0].axhline(0, c='g', ls='--'); ax[0].set_title("D3: TSDF fuses to the surface")
ax[0].set_xlabel("frame"); ax[0].set_ylabel("fused offset")
nx.draw(g, nx.spring_layout(g, seed=1), with_labels=True, ax=ax[1], node_color='lightgreen', node_size=1400, font_size=8)
ax[1].set_title("D5: queryable scene graph")
plt.suptitle("pixels -> poses -> TSDF -> semantics -> scene graph"); plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/D9
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks